# Labs 1 and 2: Naive strategies and UCB

This notebook covers Labs 1 and 2 of the course. It sets up the `seqlearn`
framework, used throughout the course, then implements the strategies of the first two
lectures: greedy, $\varepsilon$-greedy, explore-then-commit (ETC) and UCB. The experiments
check the results of Tutorials 1 and 2 numerically, then compare the strategies: regret
curves, growth in $\log t$, effect of the parameters, behavior when the gap $\Delta$ changes.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed".

The installation is described in the setup page (`SETUP.pdf`). Lab 0
(`lab00.ipynb`) covers the numpy, matplotlib and class features used below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import Agent, BernoulliBandit, UniformAgent, plot_mean, run, run_many
from seqlearn import checks
from seqlearn.plotting import COLORS, MUTED, set_style

set_style()

## 0. Getting started with the framework

An **environment** (`BernoulliBandit`, `GaussianBandit`) has attributes `n_arms`, `means`
and `gaps` (the gaps $\Delta_a$), and a method `pull(arm)` that returns a reward. Arms are
numbered from 0.

An **agent** inherits from `Agent` and implements `act()`, which returns the arm to play at
the next round. The method `update(arm, reward)`, already written, keeps track of `t`
(number of rounds played), `counts` (the $N_a(t)$) and `mu_hat` (the $\hat\mu_a(t)$) with
the incremental update seen in the lecture. Read the code of `Agent` and `UniformAgent` in
`seqlearn/bandits.py`.

`run(env, agent, horizon)` lets the agent play and returns the arms played and the rewards.
`run_many(make_env, make_agent, horizon, n_runs, seed)` repeats the experiment with
independent and reproducible seeds, and returns arrays of shape `(n_runs, horizon)`.

In [ ]:
env = BernoulliBandit([0.6, 0.4], rng=0)
agent = UniformAgent(env.n_arms, rng=1)
arms, rewards = run(env, agent, horizon=10)
print("arms played:", arms)
print("rewards    :", rewards)
print("N_a(10)    :", agent.counts)
print("mu_hat(10) :", agent.mu_hat)
print("gaps       :", env.gaps)

From now on, we work on the instance with two Bernoulli arms of means $0.6$ and $0.4$
(gap $\Delta = 0.2$), with the horizon $T = 5000$ of Tutorial 1.

In [ ]:
MEANS = [0.6, 0.4]
HORIZON = 5000
N_RUNS = 50


def make_env(rng):
    return BernoulliBandit(MEANS, rng=rng)


GAPS = make_env(None).gaps
DELTA = GAPS.max()
t = np.arange(1, HORIZON + 1)

## 1. Pseudo-regret

The pseudo-regret of a run is $\bar R_t = \sum_{s=1}^t \Delta_{A_s}$. By exercise 1 of
Tutorial 1, its expectation is the regret $R_t$.

**Question 1.** Write `pseudo_regret(arms, gaps)`, which returns the cumulative
pseudo-regret for an array `arms` of shape `(T,)` or `(n_runs, T)`.

*Hint:* `gaps[arms]` and `np.cumsum(..., axis=-1)`.

In [ ]:
def pseudo_regret(arms, gaps):
    ...  # TODO


checks.check_pseudo_regret(pseudo_regret)

**Question 2.** Run `UniformAgent` for `N_RUNS` runs. On a single figure, plot the
pseudo-regret of 5 individual runs, its mean with `plot_mean`, and the line
$t \mapsto \frac{t}{K}\sum_a \Delta_a$ obtained in Tutorial 1 (exercise 1, question 3).

In [ ]:
...  # TODO

## 2. Greedy

**Question 3.** Complete `GreedyAgent`: during the first $K$ rounds, play arms
$0, 1, \ldots, K-1$ in order; afterwards, play $\arg\max_a \hat\mu_a$, ties going to the
smallest index (this is what `np.argmax` does). The method must return a Python integer,
for instance `int(np.argmax(...))`.

In [ ]:
class GreedyAgent(Agent):
    def act(self):
        ...  # TODO


checks.check_greedy(GreedyAgent)

**Question 4.** By Tutorial 1 (exercise 2, question 4), the best arm (here arm 0) is
pulled only once if and only if the first two rewards are $0$ then $1$, which happens with
probability $(1-p)q = 0.16$.

1. Over 2000 runs of horizon 50, estimate the fraction of runs in which arm 0 is pulled
   exactly once, with a $\pm 2$ standard error interval, and check that $0.16$ lies in the
   interval.
2. Over 200 runs of horizon `HORIZON`, plot the mean pseudo-regret of greedy and the lower
   bound $(1-p)\,q\,(p-q)\,(t-1)$ of Tutorial 1 (exercise 2, question 3).

In [ ]:
...  # TODO

## 3. $\varepsilon$-greedy

**Question 5.** Complete `EpsilonGreedyAgent`: during the first $K$ rounds, play each arm
once in order; afterwards, with probability `epsilon`, play a uniform arm, and otherwise
the arm with the best empirical mean. Use `self.rng` for all random draws, so that
experiments are reproducible.

In [ ]:
class EpsilonGreedyAgent(Agent):
    def __init__(self, n_arms, epsilon, rng=None):
        self.epsilon = epsilon
        super().__init__(n_arms, rng)

    def act(self):
        ...  # TODO


checks.check_epsilon_greedy(EpsilonGreedyAgent)

## 4. Explore-then-commit

**Question 6.** Complete `ETCAgent`: during the first $mK$ rounds, play the arms in turn
($0, 1, \ldots, K-1, 0, 1, \ldots$); at the next round, choose the arm with the best
empirical mean and play it until the end.

In [ ]:
class ETCAgent(Agent):
    def __init__(self, n_arms, m, rng=None):
        self.m = m
        super().__init__(n_arms, rng)

    def reset(self):
        super().reset()
        self.committed = None

    def act(self):
        ...  # TODO


checks.check_etc(ETCAgent)

The experiments below all have the same form: run an agent for `N_RUNS` runs and compute
its pseudo-regret. The function `regret_curves` does it, for the reference instance by
default, and returns an array of shape `(n_runs, horizon)`. The function `final_stats`
returns the mean final regret and the half-width of its $\pm 2$ standard error interval.

In [ ]:
def regret_curves(make_agent, means=MEANS, horizon=HORIZON, n_runs=N_RUNS):
    arms, _ = run_many(lambda rng: BernoulliBandit(means, rng=rng), make_agent, horizon, n_runs)
    return pseudo_regret(arms, BernoulliBandit(means).gaps)


def final_stats(regret):
    final = regret[:, -1]
    return final.mean(), 2 * final.std(ddof=1) / np.sqrt(len(final))

## 5. Comparison of the naive strategies

**Question 7.** Compute $m^\star = \lceil \frac{2}{\Delta^2}\log\frac{T\Delta^2}{2} \rceil$
(Tutorial 1, exercise 3) and store it in `M_STAR`. On a single figure, plot the mean
pseudo-regret, with bands, of uniform exploration, greedy, $\varepsilon$-greedy with
$\varepsilon = 0.1$ and ETC($m^\star$), using `regret_curves`. Print for each strategy the mean final regret
$\pm 2$ standard errors, and compare the one of ETC with the bound
$\frac{2}{\Delta}\big(1 + \log\frac{T\Delta^2}{2}\big)$.

In [ ]:
...  # TODO

**Question 8.** Plot the mean final regret (with $\pm 2$ standard error bars) of ETC as a
function of $m \in \{10, 25, 50, 100, 231, 500, 1000\}$, together with the bound
$g(m) = m\Delta + T\Delta e^{-m\Delta^2/2}$ of Tutorial 1. Do the same for
$\varepsilon$-greedy as a function of
$\varepsilon \in \{0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5\}$. Use a logarithmic x-axis.

In [ ]:
...  # TODO

## 6. UCB

At round $t > K$, the strategy UCB($\alpha$) plays the arm maximizing the index
$$U_a(t) = \hat\mu_a(t-1) + \sqrt{\frac{\alpha \log t}{N_a(t-1)}} \enspace,$$
and $\alpha = 2$ gives UCB1. During the first $K$ rounds, each arm is played once, in order.

**Question 9.** Complete `UCBAgent`. The method `index()` returns the array of the $K$
indices $U_a(t)$ of the next round, and is only called once every arm has been pulled. The
method `act()` plays arms $0, 1, \ldots, K-1$ during the first $K$ rounds, then the argmax
of the index, ties going to the smallest index.

*Hint:* when `act()` is called, `self.t` rounds have been played, so the next round is
round `self.t + 1`.

In [ ]:
class UCBAgent(Agent):
    def __init__(self, n_arms, alpha=2.0, rng=None):
        self.alpha = alpha
        super().__init__(n_arms, rng)

    def index(self):
        ...  # TODO

    def act(self):
        ...  # TODO


checks.check_ucb(UCBAgent)

**Question 10.** Run UCB1 once for $1000$ rounds on the reference instance, with the loop
below, and store at each round $t > K$ the indices $U_a(t)$ and the empirical means
$\hat\mu_a(t-1)$. Plot, as functions of $t$, the two indices (solid lines), the two
empirical means (dotted lines) and the true means (horizontal lines). Print the final
counts $N_a(1000)$.

In [ ]:
env = BernoulliBandit(MEANS, rng=0)
agent = UCBAgent(env.n_arms, rng=1)
n_rounds = 1000
indices = np.full((n_rounds, env.n_arms), np.nan)
means_hat = np.full((n_rounds, env.n_arms), np.nan)
for s in range(n_rounds):
    ...  # TODO

...  # TODO

## 7. Comparison with the naive strategies

**Question 11.** On a single figure, plot the mean pseudo-regret, with bands, of
$\varepsilon$-greedy with $\varepsilon = 0.1$, ETC with $m = m^\star$ (`M_STAR`) and
UCB1, using `regret_curves`. Print for each strategy the mean final regret $\pm 2$ standard
errors. Which of the three strategies use the value of $\Delta$ or of $T$?

In [ ]:
...  # TODO

**Question 12.** Run UCB1 with horizon $T = 20000$. Plot its mean pseudo-regret with a
logarithmic x-axis, together with the bound $8\log(t)/\Delta + \Delta$ of Tutorial 2
(exercise 2). Do the same for $\varepsilon$-greedy with $\varepsilon = 0.1$.

In [ ]:
...  # TODO

## 8. The exploration parameter

**Question 13.** Plot the mean final regret (with $\pm 2$ standard error bars) of
UCB($\alpha$) as a function of $\alpha \in \{0.1, 0.25, 0.5, 1, 2, 4, 8\}$, with a
logarithmic x-axis. For $\alpha = 0.1$ and $\alpha = 2$, plot the histogram of the final
regret over the runs.

In [ ]:
...  # TODO

## 9. Changing the gap

**Question 14.** Keep the tuning of the reference instance: $\varepsilon = 0.1$,
$m = 231$ and $\alpha = 2$. For $\Delta \in \{0.05, 0.1, 0.2, 0.4\}$, run the three
strategies on the instance with means $(0.6, 0.6 - \Delta)$ and plot the mean final regret
as a function of $\Delta$, with $\pm 2$ standard error bars.

In [ ]:
...  # TODO

## Going further

**Question 15.** Consider the instance with five Bernoulli arms of means
$(0.7, 0.6, 0.5, 0.4, 0.2)$. Run UCB1 for `N_RUNS` runs of horizon `HORIZON`, and compare,
for each suboptimal arm, the mean number of pulls $N_a(T)$ with the bound
$8\log(T)/\Delta_a^2 + 1$ of Tutorial 2. Which arm contributes most to the regret?

In [ ]:
...  # TODO

**Question 16.** In Question 13, a small $\alpha$ gives a small mean regret on the reference
instance. Run UCB($\alpha$) for $\alpha \in \{0.1, 0.5, 2\}$ on the instance with means
$(0.6, 0.55)$ and horizon $T = 20000$, and plot the mean pseudo-regret with the band
`band="quantile"` of `plot_mean`.

In [ ]:
...  # TODO

**Question 17.** Implement a variant of $\varepsilon$-greedy with
$\varepsilon_t = \min(1, c/t)$, and plot its final regret as a function of
$c \in \{1, 5, 20, 100, 500\}$. Does the best $c$ depend on the gap $\Delta$? Run the
experiment again with means $(0.6, 0.5)$.

In [ ]:
...  # TODO